# BrandDeck: своя модель на A100 в Colab

Это notebook для **инференса**, обучение не требуется. Выберите среду с GPU A100 и выполняйте ячейки по порядку. Код подготовлен по официальной документации, но на вашем GPU ещё не запускался.

Модель: [Qwen2.5-VL-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct), 7B, Apache 2.0, текст + изображения. [vLLM serve](https://docs.vllm.ai/en/latest/cli/serve/) предоставляет API. [Cloudflare Quick Tunnel](https://developers.cloudflare.com/tunnel/get-started/) соединяет ваш локальный сервис с Colab.

Ноутбук скачивает веса модели. Публичный временный адрес защищается вашим API-ключом. Не публикуйте ключ или вывод ячеек с ним. Colab должен оставаться запущенным.


In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
# Используйте отдельный virtualenv, чтобы не конфликтовать с библиотеками Colab.
from pathlib import Path
ENV = Path('/content/branddeck-inference-venv')
if not (ENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ENV)], check=True)
PYTHON = str(ENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-U', 'vllm', 'huggingface_hub'], check=True)
freeze = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
Path('/content/branddeck-inference-requirements.txt').write_text(freeze)
print('Зависимости установлены; их версии сохранены в /content/branddeck-inference-requirements.txt')


## Запуск модели

Задайте свой длинный ключ. Используйте этот же ключ позже в Windows. 32K — стартовый размер контекста; реальное время и память нужно измерить на вашей A100. Ограничение четырёх изображений соответствует текущему анализатору.


In [ ]:
import getpass, os, json
MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
API_KEY = getpass.getpass('Придумайте API-ключ (не менее 24 символов): ')
assert len(API_KEY) >= 24, 'Нужен ключ длиной не менее 24 символов'
# Фиксируем точный revision модели в журнале запуска.
revision_script = 'from huggingface_hub import model_info; print(model_info(' + repr(MODEL) + ').sha)'
REVISION = subprocess.check_output([PYTHON, '-c', revision_script], text=True).strip()
config = {'model': MODEL, 'revision': REVISION, 'dtype': 'bfloat16', 'max_model_len': 32768}
Path('/content/branddeck-model.json').write_text(json.dumps(config, indent=2))
log = open('/content/branddeck-vllm.log', 'w')
command = [str(ENV / 'bin/vllm'), 'serve', MODEL, '--revision', REVISION,
           '--host', '127.0.0.1', '--port', '8001', '--api-key', API_KEY,
           '--dtype', 'bfloat16', '--max-model-len', '32768', '--max-num-seqs', '1',
           '--gpu-memory-utilization', '0.85', '--limit-mm-per-prompt', '{"image":4}',
           '--mm-processor-kwargs', '{"max_pixels":1003520}']
if 'server' in globals() and server.poll() is None:
    raise RuntimeError('Сервер уже запущен. Сначала выполните последнюю ячейку остановки.')
server = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)
print('Модель загружается. PID:', server.pid)


In [ ]:
import urllib.request, urllib.error, time
LOCAL = 'http://127.0.0.1:8001/v1'
for attempt in range(120):
    if server.poll() is not None:
        raise RuntimeError('Сервер остановился. Посмотрите /content/branddeck-vllm.log; перед публикацией удалите ключ из лога.')
    try:
        req = urllib.request.Request(LOCAL + '/models', headers={'Authorization': 'Bearer ' + API_KEY})
        with urllib.request.urlopen(req, timeout=5) as response:
            models = json.load(response)
        print('Готово:', [m['id'] for m in models['data']]); break
    except (urllib.error.URLError, TimeoutError):
        if attempt % 6 == 0: print('Ожидаем загрузку весов…')
        time.sleep(5)
else:
    raise TimeoutError('Модель не запустилась за 10 минут. Проверьте лог.')


## Проверка настоящего ответа

Проверяет текстовый запрос и JSON mode до открытия туннеля. Если этот запрос не проходит, сначала исправьте запуск модели.


In [ ]:
payload = {'model': MODEL, 'messages': [{'role': 'user', 'content': 'Return JSON with one key: status, value: ready.'}],
           'temperature': 0, 'max_tokens': 80, 'response_format': {'type': 'json_object'}}
req = urllib.request.Request(LOCAL + '/chat/completions', data=json.dumps(payload).encode(),
      headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + API_KEY})
with urllib.request.urlopen(req, timeout=120) as response:
    result = json.load(response)
print(result['choices'][0]['message']['content'])


## Временный адрес для компьютера

Эта ячейка открывает тестовый HTTPS-туннель к API. Не предназначено для production. Потоковая выдача не используется. При перезапуске туннеля адрес меняется.


In [ ]:
import re
binary = Path('/content/cloudflared')
if not binary.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', binary)
    binary.chmod(0o755)
tunnel_log = open('/content/branddeck-tunnel.log', 'w')
tunnel = subprocess.Popen([str(binary), 'tunnel', '--url', 'http://127.0.0.1:8001', '--no-autoupdate'],
                           stdout=tunnel_log, stderr=subprocess.STDOUT)
for attempt in range(30):
    if tunnel.poll() is not None: raise RuntimeError('Туннель остановился; проверьте /content/branddeck-tunnel.log')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/branddeck-tunnel.log').read_text())
    if match:
        PUBLIC = match.group(); break
    time.sleep(2)
else:
    raise TimeoutError('Не получен адрес туннеля')
print('Адрес API:', PUBLIC + '/v1')
# Проверяем, что запрос БЕЗ ключа отвергается.
try:
    urllib.request.urlopen(PUBLIC + '/v1/models', timeout=20)
    tunnel.terminate()
    raise RuntimeError('API без защиты. Туннель остановлен.')
except urllib.error.HTTPError as error:
    if error.code not in (401, 403): raise
    print('Авторизация включена.')


## Подключение BrandDeck в PowerShell на вашем компьютере

Закройте прежний локальный сервер BrandDeck и запустите заново из **того же PowerShell**, где заданы переменные. В интерфейсе снимите галочку «Без модели». Для повторного анализа шаблона с моделью используйте `analyze --force`.

```powershell
Set-Location 'C:\Users\Serg\Desktop\лцт\lct_vk'
$env:INFERENCE_BASE_URL='https://АДРЕС-ИЗ-ПРЕДЫДУЩЕЙ-ЯЧЕЙКИ/v1'
$env:INFERENCE_MODEL='Qwen/Qwen2.5-VL-7B-Instruct'
$env:INFERENCE_API_KEY='ВАШ-КЛЮЧ'
$env:INFERENCE_VISION='1'
$env:INFERENCE_TIMEOUT='120'
$env:INFERENCE_MAX_RETRIES='1'
.\.venv\Scripts\python.exe -m slide_agent --workspace slide-workspace/dataset serve
```

Откройте http://127.0.0.1:8000. В манифесте результат должен иметь `planner_mode=inference`; `offline`/`fallback` означает, что модель не сформировала принятый план. Успех подключения ещё не доказывает качество слайдов. Сохраните model revision, версии зависимостей и время контрольного прогона.


## Остановка

Выполните после работы, чтобы освободить GPU и закрыть временный доступ.


In [ ]:
for name in ('tunnel', 'server'):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
print('Сервер и туннель остановлены.')
